In [1]:
import os

# Check if D drive exists
if os.path.exists("D:/"):
    print("✅ D: drive found!")
    print(os.listdir("D:/"))
else:
    print("❌ No D: drive")
    # Use this path instead
    save_path = "C:/Users/user/Desktop/urdu-emotion-project/my-urdu-emotion-model"
    print(f"Will use: {save_path}")

✅ D: drive found!
['$RECYCLE.BIN', 'DumpStack.log.tmp', 'my-urdu-emotion-model', 'System Volume Information']


In [2]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

save_path = "D:/my-urdu-emotion-model"

tokenizer = AutoTokenizer.from_pretrained(save_path)

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=5
)

weights_path = f"{save_path}/model_weights.pt"
model.load_state_dict(torch.load(weights_path, map_location="cpu"))
model.eval()

print("✅ Model loaded from D: drive!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded from D: drive!


In [3]:
import subprocess
subprocess.run(["pip", "install", "gradio"])
print("Gradio installed!")

Gradio installed!


In [4]:
from transformers import pipeline
import gradio as gr

# Create classifier
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

# Emotion display with emojis
emotion_display = {
    "happy":   "😊 Happy — خوشی",
    "sad":     "😢 Sad — اداسی",
    "angry":   "😠 Angry — غصہ",
    "fear":    "😨 Fear — ڈر",
    "neutral": "😐 Neutral — عام"
}

def detect_emotion(urdu_text):
    if not urdu_text.strip():
        return "Please enter some Urdu text — کچھ اردو لکھیں"
    
    result = classifier(urdu_text)[0]
    emotion = result['label']
    confidence = round(result['score'] * 100, 1)
    display = emotion_display.get(emotion, emotion)
    
    output = f"""
Detected Emotion: {display}
Confidence: {confidence}%
{"🟢 High confidence" if confidence > 80 else "🟡 Medium confidence" if confidence > 60 else "🔴 Low confidence"}
    """
    return output.strip()

# Build interface
demo = gr.Interface(
    fn=detect_emotion,
    inputs=gr.Textbox(
        placeholder="یہاں اردو جملہ لکھیں...",
        label="Urdu Text — اردو متن",
        lines=3
    ),
    outputs=gr.Textbox(
        label="Emotion Result — جذبات",
        lines=5
    ),
    title="Urdu Emotion Detector",
    description="Enter any Urdu sentence and AI will detect the emotion. Built using XLM-RoBERTa.",
    examples=[
        ["میں بہت خوش ہوں"],
        ["مجھے بہت دکھ ہوا"],
        ["مجھے بہت غصہ آ رہا ہے"],
        ["مجھے ڈر لگ رہا ہے"],
        ["آج معمول کا دن تھا"]
    ],
    theme=gr.themes.Soft()
)

print(" Launching demo...")
demo.launch(share=True)

C:\Users\user\miniconda3\envs\urdu-emotion\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


 Launching demo...
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [5]:
# Fix the label mapping in the model config
model.config.id2label = {
    0: "happy",
    1: "sad", 
    2: "angry",
    3: "fear",
    4: "neutral"
}
model.config.label2id = {
    "happy": 0,
    "sad": 1,
    "angry": 2,
    "fear": 3,
    "neutral": 4
}

# Restart demo with fixed labels
demo.close()

demo2 = gr.Interface(
    fn=detect_emotion,
    inputs=gr.Textbox(
        placeholder="یہاں اردو جملہ لکھیں...",
        label="Urdu Text — اردو متن",
        lines=3
    ),
    outputs=gr.Textbox(
        label="Emotion Result — جذبات",
        lines=5
    ),
    title="🇵🇰 Urdu Emotion Detector",
    description="Enter any Urdu sentence and AI will detect the emotion. Built using XLM-RoBERTa.",
    examples=[
        ["میں بہت خوش ہوں"],
        ["مجھے بہت دکھ ہوا"],
        ["مجھے بہت غصہ آ رہا ہے"],
        ["مجھے ڈر لگ رہا ہے"],
        ["آج معمول کا دن تھا"]
    ],
    theme=gr.themes.Soft()
)

demo2.launch(share=False)
print("✅ Demo relaunched with correct labels!")

Closing server running on port: 7860


C:\Users\user\miniconda3\envs\urdu-emotion\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


✅ Demo relaunched with correct labels!
Created dataset file at: .gradio\flagged\dataset1.csv
